# Current prepared XYZ: full-year waveforms

Generate two independent sources over 365.25 days at 5 s cadence. Each padded
block's retarded geometry and TDI operators are shared by both sources, then
released. The two retained XYZ waveforms use about 303 MB; temporary caches
are bounded by one block. Run all cells with the current checkout installed
as `pip install -e ".[orbits,notebook]"` in Python 3.12+.

In [ ]:
from pathlib import Path
from dataclasses import replace
from time import perf_counter
import hashlib
import importlib.metadata as metadata
import inspect
import json
import os
import platform
import subprocess
import sys

# Load the current checkout from any directory inside this repository.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src/egb_jax_eccentric").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Run this notebook from a directory inside the repository")
sys.path.insert(0, str(ROOT / "src"))
os.environ["JAX_PLATFORMS"] = "cpu"
import numpy as np
import jax
jax.config.update("jax_enable_x64", True)

from egb_jax_eccentric import (
    EccentricBinaryParams, LINK_LABELS, default_lisaorbits,
    state_from_lisaorbits, precompute_jax_link_geometry,
    eccentric_links_jax, prepare_xyz_from_links, xyz_from_links,
)
assert Path(inspect.getfile(prepare_xyz_from_links)).is_relative_to(ROOT), (
    "Restart the kernel to load this checkout")

environment = {
    "python": platform.python_version(), "platform": platform.platform(),
    "devices": [str(device) for device in jax.devices()],
    "jax_float64": jax.config.x64_enabled,
    "versions": {name: metadata.version(name) for name in
                 ("numpy", "scipy", "jax", "jaxlib", "pytdi", "lisaorbits")},
    "tdi_source_sha256": hashlib.sha256(
        Path(inspect.getfile(prepare_xyz_from_links)).read_bytes()).hexdigest(),
}
print(json.dumps(environment, indent=2))

def build_cache(times, orbit, measurement_order=3, delay_order=3):
    tick = perf_counter()
    state = state_from_lisaorbits(orbit, times, frame="ecliptic")
    geometry = precompute_jax_link_geometry(state, orbits=orbit, frame="ecliptic")
    delays = {label: geometry["length"][i]
              for i, label in enumerate(LINK_LABELS)}
    geometry_s = perf_counter() - tick
    tick = perf_counter()
    tdi = prepare_xyz_from_links(state, generation=2,
        measurement_order=measurement_order, delay_order=delay_order,
        link_delays=delays)
    tdi_s = perf_counter() - tick
    return state, geometry, delays, tdi, np.array([geometry_s, tdi_s])

def prepared_waveform(source, geometry, tdi):
    # This public wrapper materializes host arrays, synchronizing JAX.
    links = eccentric_links_jax(source, geometry, batch_size=1,
        physics_mode="1pn_periastron", evolution_mode="fixed")
    return tdi(links)

def make_source(frequency_hz=1e-3, eccentricity=.3):
    return EccentricBinaryParams(
        mean_motion=np.pi * frequency_hz, eccentricity=eccentricity,
        m1_solar=.6, m2_solar=.4, distance_m=3.085677581491367e19,
        beta=.2, lambda_=.4, psi=.3, inclination=.8, phi0=.2, fdot=0.,
    )

In [ ]:
dt, duration_days = 5., 365.25
samples = int(round(duration_days * 86400 / dt))
block, halo = 65536, 128
orbit = default_lisaorbits("esa-trailing")
sources = [make_source(eccentricity=.3), make_source(eccentricity=.4)]

# Use the same padded shape in every block, including the last one.
benchmark_start = perf_counter()
warm_times = np.arange(-halo, block + halo) * dt
warm_state, warm_geometry, warm_delays, warm_tdi, first_setup = build_cache(warm_times, orbit)
tick = perf_counter()
warm_xyz = prepared_waveform(sources[0], warm_geometry, warm_tdi)
warmup_s = perf_counter() - tick
del warm_xyz
print(f"First waveform call (includes JAX compilation): {warmup_s:.3f} s")

In [ ]:
xyz_by_source = [{ch: np.empty(samples) for ch in "XYZ"} for _ in sources]
setup = np.zeros(2)
waveform_s = np.zeros(len(sources))
for start in range(0, samples, block):
    keep = min(block, samples - start)
    if start == 0:
        state, geometry, delays, tdi, block_setup = (
            warm_state, warm_geometry, warm_delays, warm_tdi, first_setup)
        del warm_state, warm_geometry, warm_delays, warm_tdi
    else:
        times = (start + np.arange(-halo, block + halo)) * dt
        state, geometry, delays, tdi, block_setup = build_cache(times, orbit)
    setup += block_setup
    assert max(np.max(values) for values in delays.values()) < 12.
    for index, source in enumerate(sources):
        tick = perf_counter()
        block_xyz = prepared_waveform(source, geometry, tdi)
        waveform_s[index] += perf_counter() - tick
        for channel in "XYZ":
            values = block_xyz[channel][halo:halo + keep]
            assert np.isfinite(values).all()
            assert np.all(values.imag == 0), "Cannot discard nonzero imaginary samples"
            xyz_by_source[index][channel][start:start + keep] = values.real
    del state, geometry, delays, tdi, block_xyz

print(f"Samples/channel: {samples:,}; retained duration: {samples * dt / 86400:.2f} days")
print(f"Shared geometry preparation: {setup[0]:.3f} s")
print(f"Shared TDI preparation: {setup[1]:.3f} s")
print(f"Shared preparation total: {setup.sum():.3f} s")
for source, seconds in zip(sources, waveform_s):
    print(f"e={source.eccentricity:g}: warmed full-year waveform + TDI {seconds:.3f} s")
print(f"Complete run including setup, first call, checks and copies: {perf_counter() - benchmark_start:.3f} s")
print({channel: values.shape for channel, values in xyz_by_source[0].items()})

These are CPU float64 **fixed-evolution** waveforms with 1PN periastron
advance, native ESA-trailing retarded link geometry in ecliptic J2000, and
frequency-unit generation-2 TDI. The same light travel times feed links and
TDI. Geometry/operator caches are independent of source parameters.

The 5 s cadence and interpolation orders 3/3 specify this timing workload;
they are not an accuracy-convergence claim or the full referee fitting-factor
search. No waveform files are saved. Rebuild the cache when the grid, orbit
or interpolation settings change.

Access a waveform with `xyz_by_source[0]["X"]`; retained sample `i` is at
`i * dt` seconds. Per-source timings sum synchronized block calls and exclude
shared preparation, first-call JIT work, checks and copies. They are one-run
demonstration times. For three-repeat medians, use `prepared_xyz_timing.ipynb`.

The 128-sample halo covers this generation/order/cadence and the checked
12 s arm-delay bound. Reassess padding when those settings change. A later
full-year source traversal must prepare blocks again unless its caches have
been retained; the notebook shares them within each block.